## Mockup test 1

In [ ]:
from ui_components import (
    make_node,
    make_arrow,
    render_node,
    render_arrow_shape
)

import plotly.graph_objects as go
import numpy as np

flux_values = np.concatenate([
    np.linspace(0, 1, 50),   # low → high
    np.linspace(1, 0, 50)    # high → low
])

frames = []

for f in flux_values:
    node_A = make_node("Acetyl-CoA", 0.4, 0.5, 0.5 + f)
    node_B = make_node("Citrate", 0.6, 0.5, 0.3 + 0.5*f)
    arrow = make_arrow(node_A["x"], node_A["y"], node_B["x"], node_B["y"], f)

    frame = go.Frame(
        data=[render_node(node_A), render_node(node_B)],
        layout=go.Layout(shapes=[render_arrow_shape(arrow)])
    )
    frames.append(frame)

fig = go.Figure(frames=frames)

fig.add_trace(frames[0].data[0])
fig.add_trace(frames[0].data[1])
fig.update_layout(shapes=frames[0].layout.shapes)

fig.update_layout(
    updatemenus=[{
        "type": "buttons",
        "buttons": [
            {"label": "Play", "method": "animate", "args": [None, {"frame": {"duration": 80}, "fromcurrent": True}]},
            {"label": "Pause", "method": "animate", "args": [[None], {"frame": {"duration": 0}, "mode": "immediate"}]}
        ]
    }]
)

fig.update_layout(
    plot_bgcolor="white",
    paper_bgcolor="white"
)

fig.update_xaxes(range=[0,1], visible=False)
fig.update_yaxes(range=[0,1], visible=False)

fig.show()


## Mockup test 2

In [ ]:
def step(glu_in, fat_in, atp_draw, AcCoA, Cit, dt=0.05):
    # toy kinetics, tweak later
    v_accoa_prod = 0.8*glu_in + 0.6*fat_in
    v_cit_prod   = 0.7*AcCoA
    v_accoa_use  = 0.5*atp_draw
    v_cit_use    = 0.2*atp_draw

    dAcCoA = v_accoa_prod - v_accoa_use - 0.3*AcCoA
    dCit   = v_cit_prod   - v_cit_use   - 0.2*Cit

    AcCoA_next = AcCoA + dAcCoA*dt
    Cit_next   = Cit   + dCit*dt

    # define "flux" as AcCoA→Citrate production
    flux = max(0.0, v_cit_prod)

    return AcCoA_next, Cit_next, flux


In [ ]:
glu_in  = 1.0   # glucose input rate
fat_in  = 0.5   # fat input rate
atp_draw = 0.8  # ATP draw rate

T = 100
AcCoA = 0.5
Cit   = 0.3

states = []
for _ in range(T):
    AcCoA, Cit, flux = step(glu_in, fat_in, atp_draw, AcCoA, Cit)
    states.append((AcCoA, Cit, flux))


In [ ]:
frames = []

for AcCoA, Cit, flux in states:
    node_A = make_node("Acetyl-CoA", 0.4, 0.5, AcCoA)
    node_B = make_node("Citrate",    0.6, 0.5, Cit)
    arrow  = make_arrow(node_A["x"], node_A["y"], node_B["x"], node_B["y"], flux)

    frame = go.Frame(
        data=[render_node(node_A), render_node(node_B)],
        layout=go.Layout(shapes=[render_arrow_shape(arrow)])
    )
    frames.append(frame)


In [ ]:
fig = go.Figure(frames=frames)

fig.add_trace(frames[0].data[0])
fig.add_trace(frames[0].data[1])
fig.update_layout(shapes=frames[0].layout.shapes)

fig.update_layout(
    updatemenus=[{
        "type": "buttons",
        "buttons": [
            {"label": "Play", "method": "animate", "args": [None, {"frame": {"duration": 80}, "fromcurrent": True}]},
            {"label": "Pause", "method": "animate", "args": [[None], {"frame": {"duration": 0}, "mode": "immediate"}]}
        ]
    }]
)

fig.update_layout(
    plot_bgcolor="white",
    paper_bgcolor="white"
)

fig.update_xaxes(range=[0,1], visible=False)
fig.update_yaxes(range=[0,1], visible=False)

fig.show()

## Mockup 3

In [ ]:
from ipywidgets import interact, FloatSlider

glu_slider = FloatSlider(value=1.0, min=0.0, max=2.0, step=0.1, description='Glucose')
fat_slider = FloatSlider(value=0.5, min=0.0, max=2.0, step=0.1, description='Fat')
atp_slider = FloatSlider(value=0.8, min=0.0, max=2.0, step=0.1, description='ATP draw')


In [ ]:
def run_animation(glu_in, fat_in, atp_draw):

    # --- ODE trajectory ---
    AcCoA = 0.5
    Cit   = 0.3
    states = []

    for _ in range(100):
        AcCoA, Cit, flux = step(glu_in, fat_in, atp_draw, AcCoA, Cit)
        states.append((AcCoA, Cit, flux))

    # --- Build frames ---
    frames = []
    for AcCoA, Cit, flux in states:
        node_A = make_node("Acetyl-CoA", 0.4, 0.5, AcCoA)
        node_B = make_node("Citrate",    0.6, 0.5, Cit)
        arrow  = make_arrow(node_A["x"], node_A["y"], node_B["x"], node_B["y"], flux)

        frame = go.Frame(
            data=[render_node(node_A), render_node(node_B)],
            layout=go.Layout(shapes=[render_arrow_shape(arrow)])
        )
        frames.append(frame)

    # --- Figure ---
    fig = go.Figure(frames=frames)

    fig.add_trace(frames[0].data[0])
    fig.add_trace(frames[0].data[1])
    fig.update_layout(shapes=frames[0].layout.shapes)

    fig.update_layout(
        updatemenus=[{
            "type": "buttons",
            "buttons": [
                {"label": "Play", "method": "animate", "args": [None, {"frame": {"duration": 80}, "fromcurrent": True}]},
                {"label": "Pause", "method": "animate", "args": [[None], {"frame": {"duration": 0}, "mode": "immediate"}]}
            ]
        }]
    )

    fig.update_layout(plot_bgcolor="white", paper_bgcolor="white")
    fig.update_xaxes(range=[0,1], visible=False)
    fig.update_yaxes(range=[0,1], visible=False)

    fig.show()


In [ ]:
interact(
    run_animation,
    glu_in=glu_slider,
    fat_in=fat_slider,
    atp_draw=atp_slider
)


## Mockup 5 (real-time)

In [ ]:
from dash import Dash, dcc, html, Output, Input
import plotly.graph_objects as go
import numpy as np

from ui_components import make_node, make_arrow, render_node, render_arrow_shape

app = Dash(__name__)

# ---------------------------
# KINETIC MODEL
# ---------------------------
def step(glu_in, fat_in, atp_draw, AcCoA, Cit, dt=0.05):
    v_accoa_prod = 0.8*glu_in + 0.6*fat_in
    v_cit_prod   = 0.7*AcCoA

    v_accoa_use  = 0.5*atp_draw
    v_cit_use    = 0.2*atp_draw

    dAcCoA = v_accoa_prod - v_accoa_use - 0.3*AcCoA
    dCit   = v_cit_prod   - v_cit_use   - 0.2*Cit

    AcCoA_next = AcCoA + dAcCoA*dt
    Cit_next   = Cit   + dCit*dt

    flux = max(0.0, v_cit_prod)
    return AcCoA_next, Cit_next, flux

# initial state
AcCoA = 0.5
Cit   = 0.3

# ---------------------------
# DASH LAYOUT
# ---------------------------
app.layout = html.Div([
    html.H2("Real-Time Metabolic Animation"),

    dcc.Slider(id="glu", min=0, max=2, step=0.1, value=1.0),
    html.Div("Glucose input"),

    dcc.Slider(id="fat", min=0, max=2, step=0.1, value=0.5),
    html.Div("Fat input"),

    dcc.Slider(id="atp", min=0, max=2, step=0.1, value=0.8),
    html.Div("ATP draw"),

    dcc.Graph(id="graph"),

    # interval for real-time updates
    dcc.Interval(id="tick", interval=50, n_intervals=0)
])

# ---------------------------
# CALLBACK: real-time update
# ---------------------------
@app.callback(
    Output("graph", "figure"),
    Input("tick", "n_intervals"),
    Input("glu", "value"),
    Input("fat", "value"),
    Input("atp", "value")
)
def update_graph(n, glu_in, fat_in, atp_draw):
    global AcCoA, Cit

    # ODE step
    AcCoA, Cit, flux = step(glu_in, fat_in, atp_draw, AcCoA, Cit)

    # build nodes + arrow
    node_A = make_node("Acetyl-CoA", 0.4, 0.5, AcCoA)
    node_B = make_node("Citrate",    0.6, 0.5, Cit)
    arrow  = make_arrow(node_A["x"], node_A["y"], node_B["x"], node_B["y"], flux)

    # build figure
    fig = go.Figure()

    fig.add_trace(render_node(node_A))
    fig.add_trace(render_node(node_B))
    fig.update_layout(shapes=[render_arrow_shape(arrow)])

    fig.update_xaxes(range=[0,1], visible=False)
    fig.update_yaxes(range=[0,1], visible=False)
    fig.update_layout(plot_bgcolor="white", paper_bgcolor="white")

    return fig

# ---------------------------
# RUN
# ---------------------------
#import webbrowser

if __name__ == "__main__":
    port = 8050
    url = f"http://127.0.0.1:{port}/"
    #print("Opening:", url)
    #webbrowser.open(url)
    app.run_server(debug=True, port=port)
